<a href="https://colab.research.google.com/github/ELIXIREstonia/2026-05-25-Python/blob/main/03_exploratory_visualisation_with_seaborn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Data Visualisation with Seaborn and Matplotlib

This workbook is mainly about seaborn. Seaborn is a high-level visualisation library that works naturally with pandas DataFrames. Matplotlib is still important because seaborn plots are built on top of Matplotlib objects, so we use Matplotlib for figure size, titles, labels, subplots, and saving figures.

The aim is not to memorise every plot type. The aim is to choose a suitable plot for a question, make it readable, and describe what the plot shows without overclaiming.


## Learning goals

By the end of this notebook, you should be able to:

- choose common plot types for numeric and categorical variables
- use seaborn arguments such as `data`, `x`, `y`, `hue`, `style`, `size`, `row`, and `col`
- use Matplotlib `fig` and `ax` objects to adjust labels and titles
- create distribution plots, box plots, scatter plots, bar plots, and faceted plots
- write a short interpretation of a plot
- prepare for exploratory data analysis in the follow-up statistics course


## Setup and data

We use the same `Islander_data` dataset as in the pandas workbook. Run the setup cell first.

The setup cell also creates a few plotting-friendly helper columns:

- `pd.Categorical(...)` sets a clear order for drug codes on plot axes and legends
- `.astype("category")` marks dosage as grouped labels rather than continuous numbers
- `.map({"H": "Happy", "S": "Sad"})` converts short codes into readable labels
- `pd.cut()` turns numeric ages into age-group labels
- `sns.set_theme()` sets the default visual style for seaborn plots

In [ ]:
# pathlib helps us use a local file path when the dataset is available in this repository.
from pathlib import Path

# pandas handles tabular data; matplotlib and seaborn handle plotting.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Set a clean default visual style for all seaborn plots in this notebook.
sns.set_theme(style="whitegrid", context="notebook")

# Prefer the local dataset, but keep an online fallback for Colab.
DATA_PATH = Path("data/Islander_data.csv")
DATA_URL = "https://raw.githubusercontent.com/steveahnahn/anti-anxiety-memory-test/main/Islander_data.csv"
source = DATA_PATH if DATA_PATH.exists() else DATA_URL
islanders = pd.read_csv(source)

# Create plotting-friendly categorical/helper columns.
islanders["Drug"] = pd.Categorical(islanders["Drug"], categories=["S", "A", "T"], ordered=True)
islanders["Dosage"] = islanders["Dosage"].astype("category")
islanders["memory_priming"] = islanders["Happy_Sad_group"].map({"H": "Happy", "S": "Sad"})
islanders["age_group"] = pd.cut(
    islanders["age"],
    bins=[0, 35, 50, 65, 120],
    labels=["25-35", "36-50", "51-65", "66+"],
)

islanders.head()

## First look before plotting

Before making a plot, inspect the data. This helps you avoid plotting the wrong column, missing values, or category codes you do not understand.


In [ ]:
# Check table size, column types, and numeric summaries before plotting.
print(islanders.shape)
display(islanders.dtypes)
display(islanders[["age", "Mem_Score_Before", "Mem_Score_After", "Diff"]].describe())

# 1. The seaborn pattern

Most seaborn functions follow the same pattern:

```python
sns.function_name(data=my_dataframe, x="column_for_x", y="column_for_y")
```

Add visual grouping with `hue`, `style`, or `size`. For many plots, add facets with `row` or `col` to create small multiples.

## Matplotlib plot control primer

Seaborn creates the main plot, but Matplotlib controls the figure around it.

In the examples below:

- `fig, ax = plt.subplots(...)` creates a figure (`fig`) and one plotting area (`ax`)
- `ax=...` tells seaborn where to draw
- `ax.set(...)` changes the title and axis labels
- `ax.axhline(...)` or `ax.axvline(...)` adds a reference line
- `plt.tight_layout()` adjusts spacing so labels and legends fit better

You do not need to memorise all Matplotlib methods now. Focus on recognising the pattern so you can read and adapt the plotting code.

# 2. Scatter plots for relationships

Use a scatter plot when both variables are numeric and you want to see their relationship.

Here we ask: how does age relate to memory score change (`Diff`), and does the pattern differ by drug or memory priming group?


In [ ]:
# Create a figure and one plotting area.
fig, ax = plt.subplots(figsize=(7, 5))

# Draw one point per participant.
sns.scatterplot(
    data=islanders,
    x="age",                 # horizontal position
    y="Diff",                # vertical position
    hue="Drug",              # colour by drug group
    style="memory_priming",  # marker shape by priming group
    size="Dosage",           # marker size by dosage level
    sizes=(40, 130),
    alpha=0.75,
    ax=ax,
)

# Add a reference line at no change.
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Memory score change by age",
    xlabel="Age",
    ylabel="Memory score change after treatment",
)

# Move the legend outside the plot area so it does not cover points.
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()

Read the plot from simple to complex:

1. What does one point represent?
2. What are the x-axis and y-axis?
3. What do colour, marker style, and size represent?
4. Are there visible patterns or outliers?
5. What would you need to test statistically before making a strong claim?


## Practice 1: scatter plot

Create a scatter plot of `Mem_Score_Before` vs `Mem_Score_After`.

Requirements:

- use `Drug` as colour
- add a clear title and axis labels
- add a diagonal reference line where before and after scores are equal
- write one sentence about what the diagonal line helps you see


In [ ]:
# Write your solution here.
# Tip: after plotting, use ax.plot([min_value, max_value], [min_value, max_value], ...).


# 3. Distribution plots

Use a histogram or density plot to see the shape of one numeric variable. This helps before choosing statistical tests because many tests make assumptions about distributions.

In the next example, `kde=True` adds a smooth density curve on top of the histogram. `common_norm=False` keeps the density curves for each drug group scaled separately, which makes grouped distributions easier to compare visually.

In [ ]:
# Create a histogram of the outcome variable.
fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(
    data=islanders,
    x="Diff",
    hue="Drug",          # draw separate distributions by drug group
    kde=True,            # add a smooth density curve
    element="step",      # use outlines so overlapping groups are easier to see
    stat="count",
    common_norm=False,   # scale grouped density curves separately
    ax=ax,
)

# Add a vertical reference line at no change.
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Distribution of memory score change",
    xlabel="Memory score change after treatment",
    ylabel="Number of participants",
)
plt.tight_layout()

## Practice 2: distributions

Make a distribution plot for `Mem_Score_Before`. Then make another for `Mem_Score_After`.

Questions to answer:

- Are the score ranges similar before and after?
- Are there any extreme values?
- Would a summary table alone show the same information?


In [ ]:
# Write your solution here.


# 4. Categorical comparisons

Use box plots, violin plots, strip plots, or swarm plots when you compare a numeric variable across categories.

A box plot summarises median, quartiles, and potential outliers. A strip plot shows the individual observations.


In [ ]:
# Box plots summarise the distribution of Diff within each group.
fig, ax = plt.subplots(figsize=(7, 4))

sns.boxplot(
    data=islanders,
    x="Drug",
    y="Diff",
    hue="Dosage",
    ax=ax,
)

ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Memory score change by drug and dosage",
    xlabel="Drug code",
    ylabel="Memory score change",
)
plt.tight_layout()

In [ ]:
# Strip plots show individual observations.
fig, ax = plt.subplots(figsize=(7, 4))

sns.stripplot(
    data=islanders,
    x="Drug",
    y="Diff",
    hue="Dosage",
    dodge=True,   # separate dosage groups within each drug group
    alpha=0.65,
    jitter=0.18,  # add small horizontal noise so overlapping points are visible
    ax=ax,
)

ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Individual memory score changes",
    xlabel="Drug code",
    ylabel="Memory score change",
)
plt.tight_layout()

## Practice 3: categorical comparison

Create a plot comparing `Diff` between happy and sad memory priming groups.

Try one of these options:

- `sns.boxplot`
- `sns.violinplot`
- `sns.stripplot`

Add `Drug` as `hue` if the plot stays readable.


In [ ]:
# Write your solution here.


# 5. Summary plots with uncertainty

Bar plots and point plots show summary statistics. In seaborn, `barplot` and `pointplot` calculate a summary from the raw data. The default summary is the mean.

The argument `errorbar="se"` asks seaborn to show the standard error around the mean. Treat it as a compact uncertainty summary, not as a full statistical test.

For statistics preparation, always remember that a summary plot hides individual data points. Pair it with a distribution or individual-point plot when possible.

In [ ]:
# Bar plots show group means by default.
fig, ax = plt.subplots(figsize=(7, 4))

sns.barplot(
    data=islanders,
    x="Drug",
    y="Diff",
    hue="Dosage",
    errorbar="se",  # show standard error around each mean
    ax=ax,
)

ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Mean memory score change by drug and dosage",
    xlabel="Drug code",
    ylabel="Mean memory score change",
)
plt.tight_layout()

In [ ]:
# Point plots are useful for comparing trends across an ordered category.
fig, ax = plt.subplots(figsize=(7, 4))

sns.pointplot(
    data=islanders,
    x="Dosage",
    y="Diff",
    hue="Drug",
    errorbar="se",  # show standard error around each mean
    dodge=True,     # separate drug groups slightly so points do not overlap
    markers="o",
    linestyles="-",
    ax=ax,
)

ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Mean memory score change across dosage levels",
    xlabel="Dosage level",
    ylabel="Mean memory score change",
)
plt.tight_layout()

## Practice 4: summary plot

Create a point plot showing mean `Diff` by `age_group`, with `Drug` as colour.

After plotting, answer:

- Which age group appears to have the highest mean change?
- Do all age groups have similar numbers of participants? Check with pandas.


In [ ]:
# Write your solution here.


## Concept test 1: choosing and reading plots

Take about 5 minutes. Focus on the reasoning, not the exact syntax.

## Multiple Choice Questions

1. **When is a scatter plot a good choice?**
   - A) When comparing two numeric variables.
   - B) When showing only one category count.
   - C) When saving a DataFrame to CSV.
   - D) When renaming columns.

2. **What does `hue="Drug"` usually do in a seaborn plot?**
   - A) Changes the plot title.
   - B) Maps values in `Drug` to different colours.
   - C) Filters the data to one drug.
   - D) Converts `Drug` to a number.

## True/False Statements

3. **A box plot shows distribution information that a bar plot often hides.**
   - True / False

4. **A strip plot can help show individual observations inside each category.**
   - True / False

## Short Answer Question

5. **What should you explain before interpreting a plot?**

<details>
<summary>Check your answers</summary>

#### Multiple Choice Questions

1. A) When comparing two numeric variables.
2. B) Maps values in `Drug` to different colours.

#### True/False Statements

3. True
4. True

#### Short Answer Question

5. Explain what one mark represents, what the axes show, and what any colour, size, style, row, or column grouping means.

</details>

# 6. Facets: one plot per group

Facets are small multiples. They are often clearer than putting too much information into one plot.


In [ ]:
# relplot can create multiple related scatter plots in one call.
g = sns.relplot(
    data=islanders,
    x="Mem_Score_Before",
    y="Mem_Score_After",
    hue="memory_priming",
    col="Drug",       # create one panel per drug group
    kind="scatter",
    height=4,
    aspect=0.9,
    alpha=0.75,
)

# Add the same before=after reference line to every panel.
for ax in g.axes.flat:
    ax.plot([20, 125], [20, 125], color="black", linewidth=1, linestyle="--")

g.set_axis_labels("Memory score before", "Memory score after")
g.set_titles("Drug {col_name}")
g.fig.suptitle("Before and after memory scores by drug", y=1.05)

## Practice 5: facets

Use `sns.catplot` or `sns.relplot` to create one plot per `Drug` or one plot per `memory_priming` group.

A good faceted plot should answer a question that would be harder to see in a single crowded plot.


In [ ]:
# Write your solution here.


# 7. Matplotlib objects: `fig` and `ax`

Seaborn draws onto Matplotlib axes. When you create `fig, ax = plt.subplots()`, you get explicit control over the plot area.

Use `ax.set()` for titles and labels. Use `fig.suptitle()` when one figure contains multiple subplots.


In [ ]:
# Create two side-by-side plotting areas that share the same y-axis scale.
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

# Left panel: distribution summary.
sns.violinplot(
    data=islanders,
    x="Drug",
    y="Diff",
    inner="quartile",
    ax=axes[0],
)
axes[0].set(title="Distribution summary", xlabel="Drug code", ylabel="Memory score change")

# Right panel: individual observations.
sns.stripplot(
    data=islanders,
    x="Drug",
    y="Diff",
    hue="memory_priming",
    dodge=True,
    alpha=0.65,
    ax=axes[1],
)
axes[1].set(title="Individual observations", xlabel="Drug code", ylabel="")

# Add a shared visual reference line to both panels.
for ax in axes:
    ax.axhline(0, color="black", linewidth=1, linestyle="--")

fig.suptitle("Two complementary views of the same outcome")
plt.tight_layout()

To save a figure, call `fig.savefig()` after creating it. Choose a meaningful file name and use `bbox_inches="tight"` if the legend sits outside the plot.


In [ ]:
# Example only. Uncomment when you want to save a figure.
# fig.savefig("memory_score_change.png", dpi=150, bbox_inches="tight")


# 8. Choosing an appropriate plot

Use this as a quick guide:

| Question | Good starting plot |
| --- | --- |
| How is one numeric variable distributed? | `histplot`, `kdeplot` |
| How do two numeric variables relate? | `scatterplot`, `relplot` |
| How does a numeric value differ by category? | `boxplot`, `violinplot`, `stripplot` |
| How do group means compare? | `barplot`, `pointplot` |
| Does the same pattern hold across groups? | `relplot(..., col=...)`, `catplot(..., col=...)` |

A polished plot is not automatically a good plot. A good plot answers a clear question.


# Independent task: visual exploration

Create one clear visual analysis of the `Islander_data` dataset.

Requirements:

1. Start with a question, such as:
   - Does memory score change differ by drug?
   - Does dosage appear related to memory score change?
   - Are before and after scores related in the same way for each drug?
   - Does memory priming group appear to matter?
2. Choose a plot type that matches your question.
3. Use at least two variables, and use `hue`, `style`, or `col` only when it improves the plot.
4. Add a meaningful title and axis labels.
5. Write 3 to 5 sentences interpreting the plot.
6. Include one limitation or one follow-up question that would need statistics.

For self-study, make a second version of the same plot with a different seaborn function. Compare which version communicates better.


In [ ]:
# Start your independent visualisation here.
# Suggested workflow:
# 1. Write your question in a markdown cell.
# 2. Create the plot.
# 3. Add labels and title.
# 4. Write your interpretation.


## Final concept test: exploratory visualisation

Take about 5 minutes. These questions check whether you can connect a question to a plot choice.

## Multiple Choice Questions

1. **Which plot type is a good first choice for inspecting the distribution of `Diff`?**
   - A) `sns.histplot()`
   - B) `sns.scatterplot()` with two unrelated columns
   - C) `pd.merge()`
   - D) `df.info()`

2. **Which plot type is suitable for comparing numeric `Diff` values across drug groups?**
   - A) Box plot, violin plot, or strip plot.
   - B) Line plot with no grouping variable.
   - C) CSV export.
   - D) Data type conversion.

## True/False Statements

3. **Faceting with `col=` can be clearer than adding more colours when one plot becomes crowded.**
   - True / False

4. **A visual pattern is enough to make a final statistical claim.**
   - True / False

## Short Answer Question

5. **What is Matplotlib still useful for when most plots are created with seaborn?**

<details>
<summary>Check your answers</summary>

#### Multiple Choice Questions

1. A) `sns.histplot()`
2. A) Box plot, violin plot, or strip plot.

#### True/False Statements

3. True
4. False. A plot can show patterns and questions, but statistical claims require appropriate summaries, uncertainty estimates, tests, or models.

#### Short Answer Question

5. Matplotlib is useful for figure size, axes, labels, titles, reference lines, subplots, layout, and saving figures.

</details>

## What you should feel ready for next

You are ready for the exploratory parts of a statistics course when you can:

- make a plot directly from a pandas DataFrame
- explain what each mark in the plot represents
- choose between scatter, distribution, categorical, and summary plots
- use facets for grouped comparisons
- avoid making causal or statistical claims from a plot alone

The next step is to connect these visual patterns to statistical summaries, uncertainty, and tests.
